In [1]:
import torch
from torch import nn
from pathlib import Path
from tokenizers import Tokenizer
import torch.nn.functional as F

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"

In [5]:
device

'cuda'

In [6]:
#Data

In [7]:
# #Collab setup

# data_path = Path('/kaggle/working/data')
# data_path.mkdir(exist_ok=True)
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
# !cp input.txt data/input.txt




In [8]:
!pip install tiktoken

In [9]:
#Datasets

# Using tinyshakespeare

with open("/kaggle/input/medask-gpt/input.txt", 'r', encoding='utf-8') as f:
    text = f.read()

####################################################################

#Using BookCorpus
# from datasets import load_dataset
# data = load_dataset('bookcorpus/bookcorpus')

In [10]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
data = torch.tensor(tokenizer.encode(text, allowed_special={"<|endoftext|>"}), dtype=torch.long)
vocab_size = tokenizer.n_vocab

In [11]:



# ###############################################################################
# #Character level tokenization

# # # here are all the unique characters that occur in this text
# chars = sorted(list(set(text)))
# vocab_size = len(chars)


# # create a mapping from characters to integers
# stoi = { ch: i for i,ch in enumerate(chars) }
# itos = { i:ch for i,ch in enumerate(chars) }
# encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
# decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string


In [12]:
#Hyperparameters

block_size = 512
batch_size = 16
embeddings_dims = 256
attn_dropout = 0.1
no_of_heads = 4 #IMP needs to be thoroughly calculated
dropout = 0.1
epochs = 100
max_lr = 1e-4
no_of_decoder_layers = 3 #IMP needs to be thoroughly calculated
attn_dropout = 0.1
weight_decay_optim = 0.01
experts=4
top_experts=2

In [13]:
# Train and test splits
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [14]:
len(data)

2582425

In [15]:
#Layer Normalization

class LayerNormalization(nn.Module):
    def __init__(
        self,
        embeddings_dims = embeddings_dims
    ):
        super().__init__()

        self.layer_norm = nn.LayerNorm(normalized_shape=embeddings_dims)

    def forward(self, x):
        return self.layer_norm(x)

In [16]:
# Text embeddings
class TextEmbeddings(nn.Module):
    def __init__(
        self,
        vocab_size = vocab_size,
        embeddings_dims = embeddings_dims
    ):
        super().__init__()
        self.embeddings_table = nn.Embedding(num_embeddings = vocab_size, embedding_dim=embeddings_dims, device=device) #Just a look up table to convert the toekns_ids to some numbers
        # nn.init.normal_(self.embeddings_table.weight.data, mean=0, std=0.02)

    def forward(self, x):
        return self.embeddings_table(x)

In [17]:
class Swish(nn.Module):
    def __init__(
        self,
        block_size: int = block_size,
        embeddings_dims: int = embeddings_dims
    ):
        super().__init__()

        self.sig = torch.nn.Sigmoid()


    def forward(self, x):
        swish = x * self.sig(x)

        return swish


In [18]:
class Feedforward(nn.Module):
    def __init__( self,
        block_size: int = block_size,
        embeddings_dims: int = embeddings_dims):
        super().__init__()
        
        self.layers = nn.Sequential(
            nn.Linear(embeddings_dims,4*embeddings_dims),
            nn.GELU(),
            nn.Linear(4*embeddings_dims,embeddings_dims)
        )
        
        
    def forward(self,x):
        return self.layers(x)

In [19]:
class SWiGLUExpertMoE(nn.Module):
    def __init__(
        self,
        block_size: int = block_size,
        embeddings_dims: int = embeddings_dims
    ):
        super().__init__()

        self.swish = Swish(block_size=block_size, embeddings_dims=embeddings_dims)
        self.linear_layer1 = nn.Linear(in_features=embeddings_dims, out_features=embeddings_dims, device=device, bias=False, dtype=torch.float32)
        self.linear_layer2 = nn.Linear(in_features=embeddings_dims, out_features=embeddings_dims, device=device, bias=False, dtype=torch.float32)
        self.linear_layer3 = nn.Linear(in_features=embeddings_dims, out_features=embeddings_dims, device=device, bias=False, dtype=torch.float32)




    def forward(self, x):
        swish_res = self.swish(self.linear_layer1(x))
        x_V = self.linear_layer2(x)
        res = torch.mul(swish_res, x_V)
        out = self.linear_layer3(res)
        return out


In [20]:
#Changing the above to accomodate noisy top-k gating
class NoisyTopkRouter(nn.Module):
    def __init__(self, dropout=dropout, embeddings_dims=embeddings_dims,experts=experts,top_experts=top_experts):
        super(NoisyTopkRouter, self).__init__()
        self.top_k = top_experts
        #layer for router logits
        self.topkroute_linear = nn.Linear(embeddings_dims, experts)
        self.noise_linear =nn.Linear(embeddings_dims, experts)

    
    def forward(self, mh_output):
        # mh_ouput is the output tensor from multihead self attention block
        logits = self.topkroute_linear(mh_output)

        #Noise logits
        noise_logits = self.noise_linear(mh_output)

        #Adding scaled unit gaussian noise to the logits
        noise = torch.randn_like(logits)*F.softplus(noise_logits)
        noisy_logits = logits + noise

        top_k_logits, indices = noisy_logits.topk(self.top_k, dim=-1)
        zeros = torch.full_like(noisy_logits, float('-inf'))
        sparse_logits = zeros.scatter(-1, indices, top_k_logits)
        router_output = F.softmax(sparse_logits, dim=-1)
        return router_output, indices


In [21]:
class SparseMoE(nn.Module):
    def __init__(self, dropout=dropout, embeddings_dims=embeddings_dims,experts=experts,top_experts=top_experts):
        super(SparseMoE, self).__init__()
        self.router = NoisyTopkRouter(dropout=dropout, embeddings_dims=embeddings_dims,experts=experts,top_experts=top_experts)
        self.experts = nn.ModuleList([Feedforward(block_size,embeddings_dims) for _ in range(experts)])
        self.top_k = top_experts

    def forward(self, x):
        gating_output, indices = self.router(x)
        final_output = torch.zeros_like(x)

        # Reshape inputs for batch processing
        flat_x = x.view(-1, x.size(-1))
        flat_gating_output = gating_output.view(-1, gating_output.size(-1))

        # Process each expert in parallel
        for i, expert in enumerate(self.experts):
            # Create a mask for the inputs where the current expert is in top-k
            expert_mask = (indices == i).any(dim=-1)
            flat_mask = expert_mask.view(-1)

            if flat_mask.any():
                expert_input = flat_x[flat_mask]
                expert_output = expert(expert_input)

                # Extract and apply gating scores
                gating_scores = flat_gating_output[flat_mask, i].unsqueeze(1)
                weighted_output = expert_output * gating_scores

                # Update final output additively by indexing and adding
                final_output[expert_mask] += weighted_output.squeeze(1)

        return final_output


In [22]:
#MoE Layer

class MoeLayer(nn.Module):
    def __init__(
        self,
        dropout = dropout,
        embeddings_dims = embeddings_dims,
        # inner_dimensional_states: int = 3072
    ):
        super().__init__()

        self.heads = nn.ModuleList([Feedforward(block_size,embeddings_dims) for _ in range(experts)])
        self.gate = nn.Linear(in_features=embeddings_dims, out_features=experts)
        # self.outputs = torch.zeros((batch_size,block_size, embeddings_size), device=device) #batch size needs to be defined because we are accessing it explicitly

    def forward(self, x):
        # mlp_weights_init = self.mlp.apply(weights_init)
        batch_size, block_size,embeddings_dims =   x.shape
        self.gate_out = self.gate(x)
        top_k_values, top_k_indices = torch.topk(self.gate_out, k=top_experts)
        probs = torch.nn.functional.softmax(top_k_values, dim=-1)
        #imp to add dim=-1 which specifies the softmax to be applied to the experts dim
        # print(top_k_indices[11])
        # print(top_k_values[20])
        # print(probs[20])
        outputs = torch.zeros(x.size(), device=device)
        out = 0
        for batch in range(batch_size):
            for i in range(block_size):
                for j in range(top_experts):
                    # print(i.shape)
                    # print('X batched shape: ', x[batch].shape)
                    # print('X shape: ', x.shape)
                    current_head_idx = top_k_indices[batch, i][j]
                    # print(top_k_indices[batch, i])
                    # print(top_k_indices[batch, i][j])
                    head_out = self.heads[current_head_idx](x[batch])
                    # print('Head out shape: ', head_out.shape)

                    # print('Softmax shape: ', torch.nn.functional.softmax(top_k_values[top_k_indices[i]]).shape)
                    # print('Head out shape: ', head_out.shape)
                    # print("Pro: ", probs.shape)
                    # print("Top K indices: ", top_k_indices.shape)
                    # print(probs[batch, top_k_indices[batch, i]])
                    # print(probs[batch, top_k_indices[batch, i]].shape)
                    # self.outputs[batch,i] = probs[batch, i]
                    # print(probs[batch, i].shape)
                    # print(probs[batch, i])
                    # print(probs[batch, i][j])
                    # outputs[batch,i] = probs[batch, i][j]
                    # print(self.outputs.shape)
                    out += head_out * probs[batch, i][j]

        return out


In [23]:

class AttentionHead(nn.Module):
    def __init__(
        self,
        attn_dropout = attn_dropout,
        embeddings_dims = embeddings_dims,
        no_of_heads = no_of_heads,
    ):
        super().__init__()

        assert (embeddings_dims % no_of_heads == 0), \
        "d_out must be divisible by num_heads"
        self.head_size = embeddings_dims // no_of_heads
        self.query = nn.Linear(in_features=embeddings_dims, out_features=self.head_size, device=device, bias=False)
        self.keys = nn.Linear(in_features=embeddings_dims, out_features=self.head_size,device=device, bias=False)
        self.values = nn.Linear(in_features=embeddings_dims, out_features=self.head_size, device=device,bias=False)
        self.dropout = nn.Dropout(p = attn_dropout)


    def forward(self, x):
        batch, block_size, embd_dims = x.shape
        k = self.keys(x)
        q = self.query(x)
        v = self.values(x)
        masked_table = torch.tril(torch.ones(block_size, block_size, device=device))
        weights = q @ torch.transpose(k, dim0=-2, dim1=-1) * (k.shape[-1] ** -0.5)
        masked_values = weights.masked_fill(masked_table[: block_size, : block_size] == 0, float('-inf'))
        weights_normalized = nn.functional.softmax(masked_values, dim=-1) #Normalize along the embeddings dimension for all the tokens
        weights_normalized = self.dropout(weights_normalized)
        out = weights_normalized @ v
        return out



In [24]:
# MHA




class MHA(nn.Module):
    def __init__(
        self,
        attn_dropout = attn_dropout,
        embeddings_dims = embeddings_dims,
        no_of_heads = no_of_heads,
    ):
        super().__init__()
        self.heads = nn.ModuleList([AttentionHead(attn_dropout=attn_dropout, embeddings_dims=embeddings_dims, no_of_heads=no_of_heads) for _ in range(no_of_heads)])
        self.dropout = nn.Dropout(p = attn_dropout)
        self.linear = nn.Linear(in_features=embeddings_dims, out_features=embeddings_dims, device=device, bias=False) # 12 (no of heads) * (batch_size) 64 = 768 -> gives out the text embeddings

    def forward(self, x):
        concat = torch.cat([head(x) for head in self.heads], dim=-1)
        linear_layer = self.linear(concat)
        out = self.dropout(linear_layer)
        return out

In [25]:
# Decoder Block

class TransformerDecoderBlock(nn.Module):
    def __init__(
        self,
        attn_dropout = attn_dropout,
        embeddings_dims = embeddings_dims,
        no_of_heads = no_of_heads,
        dropout = dropout,
        vocab_size = vocab_size
    ):
        super().__init__()

        self.mha = MHA(attn_dropout=attn_dropout, embeddings_dims=embeddings_dims, no_of_heads=no_of_heads)
        self.layer_norm1 = LayerNormalization(embeddings_dims=embeddings_dims)
        self.layer_norm2 = LayerNormalization(embeddings_dims=embeddings_dims)
        self.moe_block = SparseMoE(dropout=dropout, embeddings_dims=embeddings_dims)
        self.shortcut_drop = nn.Dropout(dropout)
    def forward(self, x):
        # x = self.mha(x)
        # x = x + self.layer_norm1(x)
        # x = x + self.mlp_block(x)
        # out = self.layer_norm2(x)
        x = x + self.mha(self.layer_norm1(x))  #Very important step -> Layer Norm on input and then passes it to the subsequent blocks
        x = self.shortcut_drop(x)
        x = x + self.moe_block(self.layer_norm2(x)) #Very important step
        shortcut = self.shortcut_drop(x)
        x = x+ shortcut
        return x

In [26]:
# Decoder Block

class DecoderModel(nn.Module):
    def __init__(
        self,
        attn_dropout = attn_dropout,
        embeddings_dims = embeddings_dims,
        no_of_heads = no_of_heads,
        block_size = block_size,
        dropout = dropout,
        no_of_decoder_layers = no_of_decoder_layers,
        vocab_size = vocab_size
    ):
        super().__init__()


        # self.positional_embeddings = nn.Parameter(torch.randn(1, block_size, embeddings_dims, device=device), requires_grad=True) #To give positional embeddings to each token of the input text, hence num_embeddings=block_size
        # torch.nn.init.normal_(self.positional_embeddings, mean=0.0, std=0.02)

        self.positional_embeddings = nn.Embedding(block_size, embeddings_dims,device = device) #To give positional embeddings to each token of the input text, hence num_embeddings=block_size
        # torch.nn.init.normal_(self.positional_embeddings, mean=0.0, std=0.02)
        self.text_embds = TextEmbeddings(vocab_size=vocab_size, embeddings_dims=embeddings_dims)
        self.linear_layer = nn.Linear(in_features=embeddings_dims, out_features=vocab_size, device=device, bias=False) # Takes in logits of dimensions- embeds_dims and converts it into dimension of vocab_size (logits in range of vocab_size)
        # self.layer_norm = LayerNormalization(embeddings_dims=embeddings_dims)
        self.decoder_layers = nn.Sequential(*[TransformerDecoderBlock(attn_dropout=attn_dropout, embeddings_dims=embeddings_dims, no_of_heads=no_of_heads, dropout=dropout, vocab_size=vocab_size) for _ in range(no_of_decoder_layers)])
        self.apply(self._init_weights)

    def _init_weights(self, module):  #Weight Initialization
            if isinstance(module, nn.Linear):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        batch_size,block_size=x.shape
        token_emb = self.text_embds(x)
        pos_emb = self.positional_embeddings(torch.arange(block_size, device=x.device))
        # x = self.text_embds(x)
        x = token_emb + pos_emb
        x = self.decoder_layers(x)
        # x = self.layer_norm(x)
        out = self.linear_layer(x)
        return out


        # x = self.text_embds(x)
        # x = x + self.positional_embeddings[:, :block_size, :]

        # # x = x + self.positional_embeddings(torch.arange(block_size, device=x.device))
        # x = self.decoder_layers(x)
        # # x = self.layer_norm(x)
        # out = self.linear_layer(x)
        # return out

In [27]:
#Instantiating the model

model = DecoderModel(attn_dropout=attn_dropout, embeddings_dims=embeddings_dims, no_of_heads=no_of_heads, block_size=block_size, dropout=dropout, no_of_decoder_layers=no_of_decoder_layers, vocab_size=vocab_size)
model = model.to(device)

In [28]:
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')


32.965144 M parameters


In [29]:
# #Printing a summary of the architecture
# !pip install torchinfo
# from torchinfo import summary
# idx, targets = get_batch('test')
# # idx = idx.to(device)
# summary(model=model,
#         input_data=idx,
#         col_names=["input_size", "output_size", "num_params", "trainable"],
#         col_width=20,
#         row_settings=["var_names"])

In [30]:
# Optimizer setup and scheduler steup

optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr)
# optimizer = torch.optim.Adam(model.parameters(), lr=max_lr, weight_decay=weight_decay_optim)
initial_iters = 2000
total_steps = 5000
eval_iters = 50

@torch.inference_mode()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            idx, targets = get_batch(split=split)
            logits = model(idx)
            batch_size, block_size, embeddings_dims = logits.shape
            logits = logits.view(batch_size*block_size, embeddings_dims) # Total tokens(words) => batch_size * block_size
            targets = targets.view(batch_size * block_size)
            loss = nn.functional.cross_entropy(logits, targets)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
#Train the  model
from tqdm import tqdm

model.train()
for step in tqdm(range(total_steps)):

    # every once in a while evaluate the loss on train and val sets
    # if (step  % eval_iters == 0 and step != 0) or step == total_steps - 1:
    #     losses = estimate_loss()
    #     print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")


    idx, targets = get_batch(split='train')
    logits = model(idx)
    batch_size, block_size, embeddings_dims = logits.shape
    logits = logits.view(batch_size*block_size, embeddings_dims)
    targets = targets.view(batch_size * block_size)
    loss = nn.functional.cross_entropy(logits, targets)

    optimizer.zero_grad(set_to_none=True)
    loss.backward(retain_graph=True)
    optimizer.step()
    # print(loss.item())
    # break
    if (step  % eval_iters == 0 and step != 0) or step == total_steps - 1:

      print(f"step {step}: train loss {loss.item():.4f}")

    # if step != 0 and (step % eval_iters == 0 or step == total_steps -1) :
    #     loss_values = estimate_loss()
    #     print("Train Loss at {} steps : {}".format(step, loss.item()), "Val Loss at {} steps : {}".format(step, loss_values['val']))

  1%|          | 52/5000 [00:11<16:54,  4.88it/s]

step 50: train loss 5.6761


  2%|▏         | 101/5000 [00:21<17:03,  4.79it/s]

step 100: train loss 5.5747


  3%|▎         | 152/5000 [00:31<16:39,  4.85it/s]

...

 48%|████▊     | 2401/5000 [08:15<09:01,  4.80it/s]

step 2400: train loss 3.6136


 49%|████▉     | 2451/5000 [08:26<08:48,  4.82it/s]

step 2450: train loss 3.4113


 50%|████▉     | 2489/5000 [08:33<08:43,  4.79it/s]

In [ ]:
loss.item()

In [ ]:
# context = torch.tensor(tokenizer.encode("First Citizen:"),dtype= torch.long, device=device)
# context = context.unsqueeze(0)
torch.save(model.state_dict(), "model.pt")

In [ ]:
context

In [ ]:
start_context = "Blumenthal D.M"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded,device=device).unsqueeze(0) #A
print("encoded_tensor.shape:", encoded_tensor.shape)

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context

    ###Input batch:
 ###tensor([[6109, 3626, 6100,  345],
        ##[6109, 1110, 6622,  257]])

    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond) ### batch, n_tokens, vocab_size

        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.multinomial(probas, num_samples=1)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [ ]:
out = generate_text_simple(
model=model,
idx=encoded_tensor,
max_new_tokens=50,
context_size=100
)

In [ ]:
out


In [ ]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)